# Intermediate 03 — Incident Response and Recovery

Move a suspicious agent run through detection, containment, independently authorized recovery, and exactly-once replay while preserving a verifiable chronology.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
ns = runpy.run_path('03_incident_recovery.py')
IncidentRun, RecoveryPlan, checkpoint_digest = (ns[name] for name in ('IncidentRun','RecoveryPlan','checkpoint_digest'))
now = datetime(2026, 9, 12, tzinfo=timezone.utc)
state = {'ticket':'T-7','operation':'close','version':4}
digest = checkpoint_digest(state)
run = IncidentRun('run-7','north','policy-v4','credential-v2')

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
assert run.detect('unexpected egress', detector='egress-monitor', now=now)
assert run.contain(capabilities={'ticket:write','network:external'}, responder='operator:lee', now=now)
assert run.propose_recovery(RecoveryPlan('checkpoint-4',digest,'policy-v4','credential-v2','automation:planner'), now=now)
run.phase.value

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
denied = run.authorize_recovery(approver='automation:planner', expected_checkpoint_digest=digest, current_policy_version='policy-v4', current_credential_version='credential-v2', now=now)
assert not denied
assert run.phase.value == 'recovery_pending'

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
approved = run.authorize_recovery(approver='operator:sam', expected_checkpoint_digest=digest, current_policy_version='policy-v4', current_credential_version='credential-v2', now=now)
assert approved and run.verify_event_chain()
assert run.commit_once('effect-T-7-close','close-ticket',now=now)
assert not run.commit_once('effect-T-7-close','close-ticket',now=now)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
{'events': len(run.events), 'chain_valid': run.verify_event_chain(), 'duplicate_effects': 0, 'phase': run.phase.value}

## 6. Exercise a second failure mode

In [ ]:
tampered = list(run.events)
original = run.events[0]
run.events[0] = type(original)(original.sequence, original.kind, 'changed', original.actor, original.observed_at, original.previous_hash, original.event_hash)
assert not run.verify_event_chain()
run.events = tampered

## 7. Production replacement

Production replacement: protected append-only audit, authenticated responders, distributed revocation, provider idempotency, encrypted evidence, tested rollback, affected-party handling, and explicit incident ownership.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.